# Phase 2: Event-Focused Temporal Graph & Hybrid Retrieval

<div style="background:#eaf2fb;border-left:6px solid #3266a8;padding:16px 20px;border-radius:10px"><strong style="font-size:20px;color:#153e66">01 · Environment and configuration</strong><p style="margin:6px 0 0;color:#456079">Install retrieval dependencies and mount Drive. Separate output directory; upstream outputs remain read-only.</p></div>

In [ ]:
# Colab: install lightweight retrieval dependencies (no LLM API needed)
import sys, subprocess, importlib.util
requirements = {'rank_bm25':'rank-bm25>=0.2.2', 'sentence_transformers':'sentence-transformers>=3.0.0', 'faiss':'faiss-cpu>=1.8.0', 'pyarrow':'pyarrow>=15'}
missing = [pkg for module,pkg in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
from pathlib import Path
import os, json, re, hashlib, warnings, time
import numpy as np
import pandas as pd
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab; configure GC1_ROOT environment variable if necessary.')
GC1 = Path(os.environ.get('GC1_ROOT','/content/drive/MyDrive/GC1'))
SNAP = GC1/'Temporal_snapshots'/'final_snapshots'
PRED = GC1/'Temporal_snapshots'/'model_training_efficient_investigation'
OUT = GC1/'Temporal_snapshots'/'graph_hybrid_retrieval_event_focused_full_v2_output'
OUT.mkdir(parents=True,exist_ok=True)
RUN_LIMIT = None  # full investigation queue; no pilot-only limit
MAX_GRAPH_HOPS=2
MAX_GRAPH_SEEDS=12
MAX_SEMANTIC_CANDIDATES=450
GRAPH_TOP_K=25
SEMANTIC_TOP_K=25
FUSED_TOP_K=30
FINAL_TOP_K=12
MAX_PER_EVIDENCE_KIND=4
MIN_EVENT_CANDIDATES=120
EMBEDDING_MODEL='sentence-transformers/all-MiniLM-L6-v2'
RERANKER_MODEL='cross-encoder/ms-marco-MiniLM-L-6-v2'
USE_PHASE1_TEXT=True
print('Input:',SNAP,' | Queue:',PRED/'triggers',' | New output:',OUT)

Mounted at /content/drive
Input: /content/drive/MyDrive/GC1/Temporal_snapshots/final_snapshots  | Queue: /content/drive/MyDrive/GC1/Temporal_snapshots/model_training_efficient_investigation/triggers  | New output: /content/drive/MyDrive/GC1/Temporal_snapshots/graph_hybrid_retrieval_event_focused_full_v2_output


<div style="background:#eaf2fb;border-left:6px solid #3266a8;padding:16px 20px;border-radius:10px"><strong style="font-size:20px;color:#153e66">02 · Load and validate the point-in-time graph</strong><p style="margin:6px 0 0;color:#456079">Use the compact validated snapshot export; do not reconstruct Phase 1.</p></div>

In [ ]:
required_files={'nodes':'nodes.parquet','edges':'edges.parquet','manifest':'snapshot_manifest.parquet','jira':'jira_states.parquet','deps':'active_dependencies.parquet'}
for name,filename in required_files.items():
    if not (SNAP/filename).is_file():
        raise FileNotFoundError(f'Missing {name}: {SNAP/filename}. Run the Notebook 1 final export.')
nodes=pd.read_parquet(SNAP/required_files['nodes'])
edges=pd.read_parquet(SNAP/required_files['edges'])
manifest=pd.read_parquet(SNAP/required_files['manifest'])
jira=pd.read_parquet(SNAP/required_files['jira'])
deps=pd.read_parquet(SNAP/required_files['deps'])
for df,cols in [(nodes,['available_at']),(edges,['available_at']),(manifest,['checkpoint_time']),(jira,['checkpoint_time','latest_status_change']),(deps,['checkpoint_time','active_from'])]:
    for col in cols:
        df[col]=pd.to_datetime(df[col],utc=True,errors='coerce',format='mixed')
assert {'node_uid','instance_id','label','available_at','source_id','source_type'}.issubset(nodes.columns)
assert {'edge_uid','instance_id','from_uid','to_uid','type','available_at'}.issubset(edges.columns)
assert {'snapshot_id','instance_id','week_number','checkpoint_time'}.issubset(manifest.columns)
assert {'instance_id','week_number','issue_key','historical_status','checkpoint_time'}.issubset(jira.columns)
assert {'instance_id','week_number','upstream_issue','downstream_issue','active_from','start_event_uid'}.issubset(deps.columns)
assert nodes.node_uid.is_unique and edges.edge_uid.is_unique
assert not manifest.duplicated(['instance_id','week_number']).any()
assert nodes.available_at.notna().all() and edges.available_at.notna().all()
assert edges.from_uid.isin(nodes.node_uid).all() and edges.to_uid.isin(nodes.node_uid).all()
print('Nodes',len(nodes),'Edges',len(edges),'Checkpoints',len(manifest),'Jira states',len(jira),'Dependencies',len(deps))

Nodes 96487 Edges 164101 Checkpoints 987 Jira states 228255 Dependencies 12684


<div style="background:#eaf2fb;border-left:6px solid #3266a8;padding:16px 20px;border-radius:10px"><strong style="font-size:20px;color:#153e66">03 · Load frozen investigation requests</strong><p style="margin:6px 0 0;color:#456079">Read the already-exported queue. Never use labels or future outcomes for retrieval.</p></div>

In [ ]:
def load_jsonl(path):
    with open(path,encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]
queue_files=[PRED/'triggers'/'validation_scheduled_investigations.jsonl',PRED/'triggers'/'exploratory_test_scheduled_investigations.jsonl']
for p in queue_files:
    if not p.is_file(): raise FileNotFoundError(f'Risk notebook queue missing: {p}')
queues={p.name.replace('_scheduled_investigations.jsonl',''):load_jsonl(p) for p in queue_files}
requests=[(part,req) for part,items in queues.items() for req in items]
assert requests, 'No investigation requests found'
assert len({r['trigger_id'] for _,r in requests})==len(requests),'Duplicate trigger IDs'
for part,r in requests:
    assert r.get('triggered') is True
    assert r.get('schema_version')=='2.0'
    assert {'instance_id','week_number','checkpoint_time','risk_predictions','graph_planner_input','trigger_id'}.issubset(r)
    assert not any(k in r for k in ['actual_label','future_affected_issues','future_new_links','horizon_end'])
    asof=pd.Timestamp(r['checkpoint_time'])
    assert asof.tzinfo is not None
    row=manifest[(manifest.instance_id==r['instance_id'])&(manifest.week_number==r['week_number'])]
    assert len(row)==1 and row.iloc[0]['checkpoint_time']==asof
print('Loaded requests:',{p:len(v) for p,v in queues.items()})
selected_requests=requests[:RUN_LIMIT] if RUN_LIMIT is not None else requests
print('Requests selected for full processing:',len(selected_requests))

Loaded requests: {'validation': 74, 'exploratory_test': 65}
Requests selected for full processing: 139


<div style="background:#eaf2fb;border-left:6px solid #3266a8;padding:16px 20px;border-radius:10px"><strong style="font-size:20px;color:#153e66">04 · Optional source-text enrichment</strong><p style="margin:6px 0 0;color:#456079">Only matching Phase 1 event UIDs with explicit historical availability may supply text. Otherwise use metadata-only evidence.</p></div>

In [ ]:
# Optional: Phase 1 event source text. Never infer historical availability from file modification times.
phase1_text={}
phase1_path=GC1/'entity_resolution_output'/'temporal_kg_v2_output'/'nodes.jsonl'
if USE_PHASE1_TEXT and phase1_path.exists():
    wanted=set(nodes.loc[nodes.label.astype(str).eq('Event'),'node_uid'].astype(str))
    with open(phase1_path,encoding='utf-8') as f:
        for line in f:
            try: rec=json.loads(line)
            except json.JSONDecodeError: continue
            uid=str(rec.get('uid') or '')
            if uid not in wanted: continue
            # Reject unverifiable or non-Jira source content.
            if str(rec.get('source_type','')).upper()!='JIRA': continue
            observed=pd.to_datetime(rec.get('observed_at'),utc=True,errors='coerce')
            if pd.isna(observed): continue
            content=str(rec.get('evidence_text') or '').strip()
            if not content:
                payload=rec.get('payload_json')
                if isinstance(payload,dict):
                    content=str(payload.get('evidence_text') or payload.get('description') or '').strip()
            if not content: continue
            phase1_text[uid]={'text':content[:1800],'observed_at':observed,'instance_id':str(rec.get('instance_id','')),
                               'source_id':str(rec.get('source_id') or '')}
else:
    print('Phase 1 source text not found or disabled; metadata-only retrieval remains supported.')
print('Verified-text candidates loaded:',len(phase1_text))

Verified-text candidates loaded: 0


## 4. Evidence construction — historical changes first

Create explicit status transitions from consecutive *past* Jira checkpoint states, newly active dependency links from verified timestamps, and source-backed KG events where text is available. Static states remain as fallback, with their evidence type clearly marked. The checkpoint date is the **observation bound**, not a fabricated exact change time.

In [ ]:
def utc(value):
    t=pd.Timestamp(value)
    return t.tz_localize('UTC') if t.tzinfo is None else t.tz_convert('UTC')
def cleanstr(v):
    if v is None: return ''
    try:
        if pd.isna(v): return ''
    except (TypeError, ValueError): pass
    return str(v)
def make_doc(eid, project, when, text, kind, source_id='', source_type='JIRA', node_uid=None,
             edge_uid=None, quality='STRUCTURED_RECORD', event_time=None, related_issues=None):
    return {'evidence_id':str(eid),'instance_id':str(project),'available_at':utc(when).isoformat(),
            'text':str(text)[:2500],'evidence_kind':str(kind),'source_id':str(source_id),
            'source_type':str(source_type),'node_uid':node_uid,'edge_uid':edge_uid,
            'evidence_quality':quality,'event_time':utc(event_time).isoformat() if event_time is not None else None,
            'related_issues':list(related_issues or [])}

def build_project_docs(project,week,asof):
    project=str(project); week=int(week); asof=utc(asof)
    doclist=[]
    visible=nodes[(nodes.instance_id==project)&(nodes.available_at<=asof)]
    for row in visible.itertuples(index=False):
        uid=str(row.node_uid); source_id=cleanstr(row.source_id); kind=str(row.label)
        extra=phase1_text.get(uid)
        verified=bool(extra and extra['instance_id']==project and extra['observed_at']<=asof
                      and extra['observed_at']<=row.available_at)
        text=(extra['text'] if verified else
              f'Jira temporal graph {kind}; source ID {source_id}; event UID {uid}; available {row.available_at.isoformat()}.')
        doclist.append(make_doc('NODE::'+uid,project,row.available_at,text,'KG_'+kind,source_id,
                    row.source_type,node_uid=uid,quality='VERIFIED_SOURCE_TEXT' if verified else 'GRAPH_METADATA_ONLY',
                    event_time=row.available_at,related_issues=[source_id] if source_id else []))

    current=jira[(jira.instance_id==project)&(jira.week_number==week)&(jira.checkpoint_time==asof)]
    earlier=jira[(jira.instance_id==project)&(jira.checkpoint_time<asof)]
    if not earlier.empty:
        earlier=earlier.sort_values('checkpoint_time').drop_duplicates('issue_key',keep='last')
        prior=dict(zip(earlier.issue_key.astype(str),earlier.historical_status.astype(str)))
    else: prior={}
    for row in current.itertuples(index=False):
        issue=str(row.issue_key); status=cleanstr(row.historical_status)
        old=prior.get(issue)
        if old is not None and old!=status:
            # Only the interval is known; do not claim the transition occurred exactly at the checkpoint.
            prev_time=earlier.loc[earlier.issue_key.astype(str)==issue,'checkpoint_time'].iloc[0]
            text=(f'Observed Jira status transition for issue {issue}: {old} -> {status}; '
                  f'change occurred after {prev_time.isoformat()} and by {asof.isoformat()}.')
            doclist.append(make_doc(f'STATUS_CHANGE::{project}::{week}::{issue}',project,asof,text,
                    'JIRA_STATUS_TRANSITION',issue,quality='DERIVED_FROM_CHECKPOINT_STATES',related_issues=[issue]))
        # Static state is lower-priority fallback, never represented as a change event.
        doclist.append(make_doc(f'JIRA_STATE::{project}::{week}::{issue}',project,asof,
              f'Jira issue {issue}: historical status at checkpoint is {status}.',
              'JIRA_CHECKPOINT_STATUS',issue,quality='STRUCTURED_STATE',related_issues=[issue]))

    active=deps[(deps.instance_id==project)&(deps.week_number==week)&(deps.checkpoint_time==asof)&(deps.active_from<=asof)]
    for row in active.itertuples(index=False):
        upstream=str(row.upstream_issue);downstream=str(row.downstream_issue)
        text=(f'Active Jira dependency: upstream issue {upstream} blocks downstream issue {downstream}; '
              f'link active since {row.active_from.isoformat()}.')
        doclist.append(make_doc(f'DEP::{project}::{week}::{upstream}::{downstream}',project,row.active_from,
            text,'ACTIVE_DEPENDENCY',row.start_event_uid,quality='STRUCTURED_LINK',
            event_time=row.active_from,related_issues=[upstream,downstream]))
        if asof-pd.Timedelta(days=14) < row.active_from <= asof:
            doclist.append(make_doc(f'NEW_DEP::{project}::{week}::{upstream}::{downstream}',project,row.active_from,
                f'Newly active blocking dependency within the last 14 days: {upstream} blocks {downstream}; '
                f'link began {row.active_from.isoformat()}.',
                'NEW_DEPENDENCY_EVENT',row.start_event_uid,quality='STRUCTURED_LINK',
                event_time=row.active_from,related_issues=[upstream,downstream]))
    ids=[x['evidence_id'] for x in doclist]
    assert len(ids)==len(set(ids)), 'Duplicate evidence IDs'
    assert all(utc(x['available_at'])<=asof and x['instance_id']==project for x in doclist)
    return doclist
print('Event-focused evidence builder ready.')

Event-focused evidence builder ready.


## 5. Risk-aware graph retrieval

Prefer actual event and dependency records over repetitive static states. Graph paths are restricted to temporally visible edges; explicit dependency evidence can reference unsampled issues but is not falsely assigned a graph path.

In [ ]:
from collections import defaultdict,deque
RISK_TERMS={'schedule':['reopened','regressed','blocked','status transition','workflow','issue','milestone'],
            'scope':['epic','reclassification','requirement','scope','change','issue'],
            'dependency':['block','dependency','upstream','downstream','chain','issue']}
def query_for_request(req):
    risks=req.get('triggered_risks') or req['graph_planner_input'].get('risk_types',[])
    hints=req['graph_planner_input'].get('entity_focus',[])
    driver_names=[]
    for rp in req.get('risk_predictions',[]):
        if rp.get('risk_type') in risks:
            for driver in rp.get('top_shap_drivers') or []:
                if isinstance(driver,dict):driver_names.append(str(driver.get('feature') or driver.get('feature_name') or ''))
    terms=sorted({t for risk in risks for t in RISK_TERMS.get(risk,[])})
    return 'Project risk investigation. '+ ' '.join(map(str,risks+hints+driver_names+terms))
def graph_retrieve(project,week,asof,docs,query,hops=MAX_GRAPH_HOPS):
    asof=utc(asof)
    vn=nodes[(nodes.instance_id==project)&(nodes.available_at<=asof)]
    ve=edges[(edges.instance_id==project)&(edges.available_at<=asof)]
    valid=set(vn.node_uid.astype(str))
    ve=ve[ve.from_uid.isin(valid)&ve.to_uid.isin(valid)]
    adjacency=defaultdict(list)
    for e in ve.itertuples(index=False):
        a,b=str(e.from_uid),str(e.to_uid)
        adjacency[a].append((b,str(e.edge_uid),str(e.type)))
        adjacency[b].append((a,str(e.edge_uid),str(e.type)))
    docmap={d['evidence_id']:d for d in docs}
    terms=set(re.findall(r'[a-z0-9_]+',query.lower()))
    # Risk-focused deterministic seeds; no label or future-outcome data.
    candidate=[(sum(t in str(d['text']).lower() for t in terms) + (3 if d['evidence_quality']=='VERIFIED_SOURCE_TEXT' else 0),d) for d in docs if d['node_uid']]
    candidate.sort(key=lambda z:(-z[0],z[1]['evidence_id']))
    seeds=[d['node_uid'] for score,d in candidate[:MAX_GRAPH_SEEDS] if score>0]
    if not seeds: seeds=[d['node_uid'] for _,d in candidate[:MAX_GRAPH_SEEDS]]
    visited={}; paths={}; frontier=deque()
    for uid in seeds:
        if uid not in visited:visited[uid]=0;paths[uid]=[];frontier.append(uid)
    while frontier:
        current=frontier.popleft()
        if visited[current]>=hops:continue
        for nxt,eid,typ in adjacency.get(current,[]):
            if nxt not in visited:
                visited[nxt]=visited[current]+1
                paths[nxt]=paths[current]+[{'edge_uid':eid,'type':typ,'from_uid':current,'to_uid':nxt}]
                frontier.append(nxt)
    result=[]
    for uid,depth in visited.items():
        doc=docmap.get('NODE::'+uid)
        if doc:
            result.append({'evidence_id':doc['evidence_id'],'graph_depth':depth,'graph_path':paths[uid],
                           'graph_score':1/(1+depth)})
    # Explicit dependency links are graph evidence even when their issues are outside the sampled node set.
    for d in docs:
        if d['evidence_kind'] in ('ACTIVE_DEPENDENCY','NEW_DEPENDENCY_EVENT'):
            result.append({'evidence_id':d['evidence_id'],'graph_depth':1,'graph_path':[], 'graph_score':1.15 if d['evidence_kind']=='NEW_DEPENDENCY_EVENT' else 0.8})
    # Include verified status transitions as direct structured graph-linked evidence.
    for d in docs:
        if d['evidence_kind']=='JIRA_STATUS_TRANSITION':
            result.append({'evidence_id':d['evidence_id'],'graph_depth':0,'graph_path':[], 'graph_score':1.05})
    result.sort(key=lambda x:(-x['graph_score'],x['evidence_id']))
    return result[:GRAPH_TOP_K]
print('Temporal traversal ready.')

Temporal traversal ready.


## 6. Balanced FAISS + BM25 retrieval

Create a deterministic candidate pool with reserved capacity for event-like evidence, verified source text, dependency links, and static-state fallback. Avoid letting hundreds of identical-time status snapshots consume the entire semantic search pool.

In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
embedder=SentenceTransformer(EMBEDDING_MODEL,device='cpu')
reranker=CrossEncoder(RERANKER_MODEL,device='cpu',max_length=384)
def tokens(text): return re.findall(r'[a-z0-9_]+',str(text).lower())
EVENT_KINDS={'JIRA_STATUS_TRANSITION','NEW_DEPENDENCY_EVENT','ACTIVE_DEPENDENCY'}
def evidence_priority(d):
    kind=d['evidence_kind']
    if d['evidence_quality']=='VERIFIED_SOURCE_TEXT': return 0
    if kind=='NEW_DEPENDENCY_EVENT': return 1
    if kind=='JIRA_STATUS_TRANSITION': return 2
    if kind=='ACTIVE_DEPENDENCY': return 3
    if kind.startswith('KG_Event'): return 4
    if kind=='JIRA_CHECKPOINT_STATUS': return 9
    return 6

def balanced_candidates(docs,asof,limit=MAX_SEMANTIC_CANDIDATES):
    asof=utc(asof)
    eligible=[d for d in docs if pd.notna(pd.to_datetime(d.get('available_at'),utc=True,errors='coerce'))
              and utc(d['available_at'])<=asof]
    eligible.sort(key=lambda d:(evidence_priority(d),-utc(d['available_at']).value,d['evidence_id']))
    # Reserve event-oriented capacity; then fill from remaining categories by priority.
    events=[d for d in eligible if d['evidence_kind'] in EVENT_KINDS or d['evidence_quality']=='VERIFIED_SOURCE_TEXT']
    picked=events[:min(limit,MIN_EVENT_CANDIDATES)]
    seen={d['evidence_id'] for d in picked}
    picked.extend(d for d in eligible if d['evidence_id'] not in seen)
    return picked[:limit]

def semantic_retrieve(query,docs,asof):
    candidates=balanced_candidates(docs,asof)
    if not candidates:return []
    texts=[d['text'] for d in candidates]
    bm25=BM25Okapi([tokens(t) or ['empty'] for t in texts])
    lexical=np.asarray(bm25.get_scores(tokens(query)),dtype=float)
    vectors=np.asarray(embedder.encode(texts,batch_size=32,normalize_embeddings=True,show_progress_bar=False),dtype='float32')
    q=np.asarray(embedder.encode([query],normalize_embeddings=True,show_progress_bar=False),dtype='float32')
    index=faiss.IndexFlatIP(vectors.shape[1]);index.add(vectors)
    sim,idx=index.search(q,min(SEMANTIC_TOP_K,len(candidates)))
    lexical_top=np.argsort(-lexical,kind='stable')[:SEMANTIC_TOP_K]
    selected=set(int(i) for i in idx[0] if i>=0)|set(int(i) for i in lexical_top)
    dense_scores={int(i):float(s) for s,i in zip(sim[0],idx[0]) if i>=0}
    result=[{'evidence_id':candidates[i]['evidence_id'],'dense_score':dense_scores.get(i,0.0),
             'bm25_score':float(lexical[i])} for i in selected]
    result.sort(key=lambda x:(-x['dense_score'],-x['bm25_score'],x['evidence_id']))
    lookup={d['evidence_id']:d for d in candidates}; asof=utc(asof)
    assert all(utc(lookup[x['evidence_id']]['available_at'])<=asof for x in result)
    return result
print('Balanced semantic and lexical retrieval ready.')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Balanced semantic and lexical retrieval ready.


## 6a. Deterministic retrieval sanity checks

Check candidate balance and chronology using a small synthetic set; these checks do not establish relevance on the real dataset.

In [ ]:
# Lightweight sanity checks for risk-aware candidate selection and time validity.
import pandas as pd
_test_docs=[
 {'evidence_id':'s1','evidence_kind':'JIRA_CHECKPOINT_STATUS','evidence_quality':'STRUCTURED_STATE','available_at':'2020-01-01T00:00:00+00:00'},
 {'evidence_id':'e1','evidence_kind':'JIRA_STATUS_TRANSITION','evidence_quality':'DERIVED_FROM_CHECKPOINT_STATES','available_at':'2020-01-02T00:00:00+00:00'},
 {'evidence_id':'future','evidence_kind':'NEW_DEPENDENCY_EVENT','evidence_quality':'STRUCTURED_LINK','available_at':'2020-01-20T00:00:00+00:00'}]
_test_selected=balanced_candidates(_test_docs,pd.Timestamp('2020-01-03',tz='UTC'),limit=2)
assert [d['evidence_id'] for d in _test_selected]==['e1','s1']
print('Candidate prioritization and future-evidence exclusion PASSED')

Candidate prioritization and future-evidence exclusion PASSED


## 7. RRF, cross-encoder and evidence diversity

Rerank fused candidates, then enforce a maximum count per evidence kind. This prevents generic checkpoint status records from crowding out event sequences and active dependencies. Relevance scores do not establish factual correctness.

In [ ]:
def fuse_rerank(query,docs,graph_hits,semantic_hits):
    lookup={d['evidence_id']:d for d in docs}
    channels={
      'graph':[x['evidence_id'] for x in sorted(graph_hits,key=lambda x:-x['graph_score'])],
      'dense':[x['evidence_id'] for x in sorted(semantic_hits,key=lambda x:-x['dense_score'])],
      'bm25':[x['evidence_id'] for x in sorted(semantic_hits,key=lambda x:-x['bm25_score'])]}
    scores=defaultdict(float);sources=defaultdict(list)
    for channel,ranked in channels.items():
        for rank,eid in enumerate(ranked,1):
            if eid not in lookup: continue
            scores[eid]+=1/(60+rank);sources[eid].append(channel)
    candidates=sorted(scores,key=lambda eid:(-scores[eid],eid))[:FUSED_TOP_K]
    if not candidates:return []
    ce=np.asarray(reranker.predict([(query,lookup[eid]['text']) for eid in candidates],
               batch_size=16,show_progress_bar=False),dtype=float).reshape(-1)
    graph_map={x['evidence_id']:x for x in graph_hits}
    result=[]
    for eid,score in zip(candidates,ce):
        row=dict(lookup[eid]);row.update({'rrf_score':float(scores[eid]),'rerank_score':float(score),
            'retrieval_channels':sources[eid],'graph_path':graph_map.get(eid,{}).get('graph_path',[]),
            'graph_depth':graph_map.get(eid,{}).get('graph_depth')})
        result.append(row)
    result.sort(key=lambda d:(-d['rerank_score'],-d['rrf_score'],d['evidence_id']))
    chosen=[];counts=defaultdict(int)
    for row in result:
        if counts[row['evidence_kind']]>=MAX_PER_EVIDENCE_KIND: continue
        chosen.append(row);counts[row['evidence_kind']]+=1
        if len(chosen)>=FINAL_TOP_K:break
    for rank,d in enumerate(chosen,1):d['rank']=rank
    return chosen
print('Diversified fusion and reranking ready.')

Diversified fusion and reranking ready.


## 8. Full-queue retrieval with evidence sufficiency and resumable exports

For every scheduled investigation, run graph traversal, balanced FAISS/BM25, RRF and cross-encoder reranking. Save each verified bundle immediately to the **new** output folder. On rerun, existing bundles are reused only when their trigger ID, checkpoint and schema match. A weak evidence bundle is still exported, but explicitly flagged. **Do not infer event relevance from a successful retrieval status.**

In [ ]:
def retrieve_one(partition,request):
    project=str(request['instance_id']);week=int(request['week_number']);asof=utc(request['checkpoint_time'])
    docs=build_project_docs(project,week,asof)
    query=query_for_request(request)
    graph_hits=graph_retrieve(project,week,asof,docs,query)
    semantic_hits=semantic_retrieve(query,docs,asof)
    evidence=fuse_rerank(query,docs,graph_hits,semantic_hits)
    for d in evidence:
        assert d['instance_id']==project and utc(d['available_at'])<=asof
    assert len({d['evidence_id'] for d in evidence})==len(evidence)
    bundle={'schema_version':'3.2','retrieval_status':'READY_FOR_AGENTIC_GRAPHRAG' if evidence else 'INSUFFICIENT_EVIDENCE',
       'trigger_id':request['trigger_id'],'snapshot_id':request['snapshot_id'],
       'instance_id':project,'week_number':week,'checkpoint_time':asof.isoformat(),
       'partition':partition,'risk_types':request.get('triggered_risks',[]),
       'risk_predictions':request['risk_predictions'],'query':query,
       'temporal_policy':'ONLY_EVIDENCE_AVAILABLE_AT_OR_BEFORE_CHECKPOINT',
       'graph_planner_input':request['graph_planner_input'],
       'retrieval_summary':{'documents_available':len(docs),'graph_hits':len(graph_hits),
          'semantic_hits':len(semantic_hits),'final_evidence':len(evidence),
          'verified_source_text':sum(d['evidence_quality']=='VERIFIED_SOURCE_TEXT' for d in evidence),
          'event_or_dependency_evidence':sum(d['evidence_kind'] in EVENT_KINDS for d in evidence),
          'evidence_kinds':sorted(set(d['evidence_kind'] for d in evidence))},
       'evidence':evidence,'investigation_caveats':[
          'Risk scores are experimental predictions, not proof of a project failure.',
          'SHAP is predictive attribution, not causal evidence.',
          'GRAPH_METADATA_ONLY and DERIVED_FROM_CHECKPOINT_STATES are not source-text quotations.',
          'A status transition inferred from consecutive checkpoints has interval timing, not an exact change timestamp.',
          'The compact snapshot excludes unverified meeting and PRD records.',
          'The next notebook must verify source provenance and claim support before explanation.']}
    return bundle


# Evidence availability is NOT the same as relevance or claim support.
EVENT_KINDS_FOR_SUFFICIENCY = {
    'JIRA_STATUS_TRANSITION', 'NEW_DEPENDENCY_EVENT',
    'ACTIVE_DEPENDENCY', 'DEPENDENCY_EVENT', 'SCOPE_EVENT'
}

def assess_evidence_sufficiency(evidence):
    kinds = [str(e.get('evidence_kind', '')) for e in evidence]
    events = sum(k in EVENT_KINDS_FOR_SUFFICIENCY for k in kinds)
    verified = sum(e.get('evidence_quality') == 'VERIFIED_SOURCE_TEXT' for e in evidence)
    if not evidence:
        status, reason = 'INSUFFICIENT_EVIDENCE', 'No evidence retrieved.'
    elif events == 0 and verified == 0:
        status, reason = 'INSUFFICIENT_EVIDENCE', 'Only static or graph metadata was retrieved.'
    else:
        status, reason = 'EVIDENCE_AVAILABLE_UNVERIFIED_RELEVANCE', 'Historical structured event/link or verified source text available; relevance and claim support not yet verified.'
    return {
        'status': status, 'reason': reason,
        'event_or_dependency_count': int(events),
        'verified_source_text_count': int(verified),
        'evidence_kind_counts': {k: kinds.count(k) for k in sorted(set(kinds))}
    }

_original_retrieve_one = retrieve_one

def retrieve_one(partition, request):
    bundle = _original_retrieve_one(partition, request)
    bundle['evidence_sufficiency'] = assess_evidence_sufficiency(bundle['evidence'])
    bundle['retrieval_status'] = ('INSUFFICIENT_EVIDENCE'
        if bundle['evidence_sufficiency']['status'] == 'INSUFFICIENT_EVIDENCE'
        else 'READY_FOR_AGENTIC_GRAPHRAG')
    return bundle

# Keep request order and identity stable. No request is silently dropped.
selected_requests = requests if RUN_LIMIT is None else requests[:RUN_LIMIT]
assert len({r['trigger_id'] for _, r in selected_requests}) == len(selected_requests)

import json, os
bundle_dir = OUT / 'retrieval_bundles'
bundle_dir.mkdir(parents=True, exist_ok=True)

def bundle_path_for(trigger_id):
    # Filename is derived from the ID hash; avoids unsafe path characters.
    digest = hashlib.sha256(str(trigger_id).encode('utf-8')).hexdigest()
    return bundle_dir / (digest + '.json')

def save_json_atomic(path, data):
    temp = path.with_name(path.name + '.partial')
    with open(temp, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False, allow_nan=False)
    os.replace(temp, path)

def load_or_retrieve(partition, request):
    path = bundle_path_for(request['trigger_id'])
    if path.exists():
        try:
            saved = json.loads(path.read_text(encoding='utf-8'))
            if (saved.get('schema_version') == '3.2'
                and saved.get('trigger_id') == request['trigger_id']
                and saved.get('snapshot_id') == request['snapshot_id']
                and saved.get('checkpoint_time') == utc(request['checkpoint_time']).isoformat()
                and saved.get('partition') == partition
                and 'evidence_sufficiency' in saved):
                return saved, True
        except (ValueError, OSError, KeyError, TypeError):
            pass
    bundle = retrieve_one(partition, request)
    save_json_atomic(path, bundle)
    return bundle, False

pilot_bundles = []  # historical variable name retained for downstream cells
reused_count = 0
for i, (partition, request) in enumerate(selected_requests, 1):
    started = time.time()
    bundle, reused = load_or_retrieve(partition, request)
    pilot_bundles.append(bundle)
    reused_count += int(reused)
    print(f"[{i}/{len(selected_requests)}] {request['trigger_id']} -> "
          f"{bundle['retrieval_status']} | evidence={len(bundle['evidence'])} "
          f"| {'reused' if reused else 'computed'} | {time.time()-started:.1f}s")
assert len(pilot_bundles) == len(selected_requests)
print('Full selected queue processed:', len(pilot_bundles), '| reused:', reused_count)


[1/139] aec0dacc1eac39d2943e -> INSUFFICIENT_EVIDENCE | evidence=8 | computed | 8.0s
[2/139] 157e5547098d1ea35ccd -> READY_FOR_AGENTIC_GRAPHRAG | evidence=10 | computed | 8.7s
[3/139] 67b3c4b34422e9148001 -> INSUFFICIENT_EVIDENCE | evidence=12 | computed | 8.6s
[4/139] 0dbd32c9c1b0f80de3f6 -> INSUFFICIENT_EVIDENCE | evidence=12 | computed | 10.3s
[5/139] ad8c84c5202d7c3bd855 -> READY_FOR_AGENTIC_GRAPHRAG | evidence=12 | computed | 9.0s
[6/139] e292d664b093fd574fad -> READY_FOR_AGENTIC_GRAPHRAG | evidence=12 | computed | 14.2s
[7/139] 45d52274cba0e5afd7c8 -> READY_FOR_AGENTIC_GRAPHRAG | evidence=12 | computed | 7.4s
[8/139] bb105922a309bbb9942d -> READY_FOR_AGENTIC_GRAPHRAG | evidence=9 | computed | 9.0s
[9/139] 12d6260c8d5cfb1b7b4a -> INSUFFICIENT_EVIDENCE | evidence=4 | computed | 6.7s
[10/139] 66f190f2b949ccca011e -> READY_FOR_AGENTIC_GRAPHRAG | evidence=12 | computed | 8.7s
[11/139] b4c4e6bcdec5f936c6f2 -> READY_FOR_AGENTIC_GRAPHRAG | evidence=12 | computed | 8.4s
[12/139] 2ebd5a03c

## 9. Inspect evidence quality and risk coverage

Review evidence kinds and insufficiency across all requests. The number of returned records alone is not a relevance metric.

In [ ]:
quality_rows=[]
for bundle in pilot_bundles:
    summary=bundle['evidence_sufficiency']
    quality_rows.append({
        'trigger_id':bundle['trigger_id'],
        'partition':bundle['partition'],
        'risk_types':','.join(bundle['risk_types']),
        'retrieval_status':bundle['retrieval_status'],
        'evidence_count':len(bundle['evidence']),
        'event_or_dependency_count':summary['event_or_dependency_count'],
        'verified_source_text_count':summary['verified_source_text_count'],
        'evidence_sufficiency':summary['status']
    })
quality_df=pd.DataFrame(quality_rows)
display(quality_df.head(15))
print('Retrieval status distribution:')
display(quality_df['retrieval_status'].value_counts())
print('Evidence type distribution:')
from collections import Counter
print(dict(Counter(e['evidence_kind'] for b in pilot_bundles for e in b['evidence'])))
if pilot_bundles:
    print('Sample top-ranked evidence:')
    display(pd.DataFrame(pilot_bundles[0]['evidence']).head(8))


,trigger_id,partition,risk_types,retrieval_status,evidence_count,event_or_dependency_count,verified_source_text_count,evidence_sufficiency
0,aec0dacc1eac39d2943e,validation,schedule,INSUFFICIENT_EVIDENCE,8,0,0,INSUFFICIENT_EVIDENCE
1,157e5547098d1ea35ccd,validation,"schedule,scope",READY_FOR_AGENTIC_GRAPHRAG,10,1,0,EVIDENCE_AVAILABLE_UNVERIFIED_RELEVANCE
2,67b3c4b34422e9148001,validation,schedule,INSUFFICIENT_EVIDENCE,12,0,0,INSUFFICIENT_EVIDENCE
3,0dbd32c9c1b0f80de3f6,validation,schedule,INSUFFICIENT_EVIDENCE,12,0,0,INSUFFICIENT_EVIDENCE
4,ad8c84c5202d7c3bd855,validation,schedule,READY_FOR_AGENTIC_GRAPHRAG,12,1,0,EVIDENCE_AVAILABLE_UNVERIFIED_RELEVANCE
5,e292d664b093fd574fad,validation,schedule,READY_FOR_AGENTIC_GRAPHRAG,12,1,0,EVIDENCE_AVAILABLE_UNVERIFIED_RELEVANCE
6,45d52274cba0e5afd7c8,validation,schedule,READY_FOR_AGENTIC_GRAPHRAG,12,1,0,EVIDENCE_AVAILABLE_UNVERIFIED_RELEVANCE
7,bb105922a309bbb9942d,validation,scope,READY_FOR_AGENTIC_GRAPHRAG,9,1,0,EVIDENCE_AVAILABLE_UNVERIFIED_RELEVANCE
8,12d6260c8d5cfb1b7b4a,validation,"dependency,schedule,scope",INSUFFICIENT_EVIDENCE,4,0,0,INSUFFICIENT_EVIDENCE
9,66f190f2b949ccca011e,validation,scope,READY_FOR_AGENTIC_GRAPHRAG,12,1,0,EVIDENCE_AVAILABLE_UNVERIFIED_RELEVANCE


Retrieval status distribution:


,count
retrieval_status,
READY_FOR_AGENTIC_GRAPHRAG,125
INSUFFICIENT_EVIDENCE,14


Evidence type distribution:
{'KG_HistoricalIssue': 60, 'KG_Event': 393, 'JIRA_STATUS_TRANSITION': 263, 'KG_CanonicalEntity': 78, 'JIRA_CHECKPOINT_STATUS': 56, 'NEW_DEPENDENCY_EVENT': 91, 'ACTIVE_DEPENDENCY': 329}
Sample top-ranked evidence:


,evidence_id,instance_id,available_at,text,evidence_kind,source_id,source_type,node_uid,edge_uid,evidence_quality,event_time,related_issues,rrf_score,rerank_score,retrieval_channels,graph_path,graph_depth,rank
0,NODE::HIST_ISSUE::SAK-I01::SAK-3113,SAK-I01,2005-12-06T17:53:55+00:00,Jira temporal graph HistoricalIssue; source ID...,KG_HistoricalIssue,SAK-3113,JIRA,HIST_ISSUE::SAK-I01::SAK-3113,None,GRAPH_METADATA_ONLY,2005-12-06T17:53:55+00:00,[SAK-3113],0.025316,-9.510154,"[dense, bm25]",[],NaN,1
1,NODE::HIST_ISSUE::SAK-I01::SAK-8535,SAK-I01,2007-02-28T12:23:47+00:00,Jira temporal graph HistoricalIssue; source ID...,KG_HistoricalIssue,SAK-8535,JIRA,HIST_ISSUE::SAK-I01::SAK-8535,None,GRAPH_METADATA_ONLY,2007-02-28T12:23:47+00:00,[SAK-8535],0.025641,-9.531974,"[dense, bm25]",[],NaN,2
2,NODE::HIST_ISSUE::SAK-I01::SAK-7031,SAK-I01,2006-10-30T21:34:25+00:00,Jira temporal graph HistoricalIssue; source ID...,KG_HistoricalIssue,SAK-7031,JIRA,HIST_ISSUE::SAK-I01::SAK-7031,None,GRAPH_METADATA_ONLY,2006-10-30T21:34:25+00:00,[SAK-7031],0.031746,-9.548423,"[dense, bm25]",[],NaN,3
3,NODE::HIST_ISSUE::SAK-I01::SAK-10626,SAK-I01,2007-07-10T06:32:08+00:00,Jira temporal graph HistoricalIssue; source ID...,KG_HistoricalIssue,SAK-10626,JIRA,HIST_ISSUE::SAK-I01::SAK-10626,None,GRAPH_METADATA_ONLY,2007-07-10T06:32:08+00:00,[SAK-10626],0.029851,-9.558331,"[dense, bm25]",[],NaN,4
4,NODE::T_41f6cceeb47197f198d55f53,SAK-I01,2012-03-27T07:00:42+00:00,Jira temporal graph Event; source ID SAK-11875...,KG_Event,SAK-11875,JIRA,T_41f6cceeb47197f198d55f53,None,GRAPH_METADATA_ONLY,2012-03-27T07:00:42+00:00,[SAK-11875],0.033776,-10.175867,"[graph, dense, bm25]","[{'edge_uid': 'L_2153cf45350a0743a5277e86', 't...",1.0,5
5,NODE::T_181b82055acf24f5fca5479a,SAK-I01,2012-03-27T12:12:25+00:00,Jira temporal graph Event; source ID SAK-8097;...,KG_Event,SAK-8097,JIRA,T_181b82055acf24f5fca5479a,None,GRAPH_METADATA_ONLY,2012-03-27T12:12:25+00:00,[SAK-8097],0.022989,-10.226470,"[dense, bm25]",[],NaN,6
6,NODE::T_18a846c4eee31411c14abc35,SAK-I01,2012-03-27T07:00:32+00:00,Jira temporal graph Event; source ID SAK-14743...,KG_Event,SAK-14743,JIRA,T_18a846c4eee31411c14abc35,None,GRAPH_METADATA_ONLY,2012-03-27T07:00:32+00:00,[SAK-14743],0.022727,-10.235389,"[dense, bm25]",[],NaN,7
7,NODE::T_2867c16c605c2bdd9448e1ba,SAK-I01,2012-03-27T06:59:23+00:00,Jira temporal graph Event; source ID SAK-11076...,KG_Event,SAK-11076,JIRA,T_2867c16c605c2bdd9448e1ba,None,GRAPH_METADATA_ONLY,2012-03-27T06:59:23+00:00,[SAK-11076],0.035438,-10.267424,"[graph, dense, bm25]","[{'edge_uid': 'L_dce7c13dfee63a42a5feef67', 't...",1.0,8


<div style="background:#eaf2fb;border-left:6px solid #3266a8;padding:16px 20px;border-radius:10px"><strong style="font-size:20px;color:#153e66">11 · Temporal, provenance and payload validation</strong><p style="margin:6px 0 0;color:#456079">Reject future evidence, foreign projects, duplicate IDs, missing source references and leaked ground-truth fields.</p></div>

In [ ]:
FORBIDDEN={'actual_label','schedule_risk_label','scope_risk_label','dependency_risk_label',
           'future_schedule_events','future_scope_events','future_new_links','horizon_end'}
def validate_bundle(b):
    asof=utc(b['checkpoint_time']);ids=set();checks=[]
    checks.append(('no forbidden top-level fields',not (set(b)&FORBIDDEN)))
    checks.append(('matching snapshot',b['snapshot_id']==f"{b['instance_id']}::W{int(b['week_number']):02d}"))
    checks.append(('bounded evidence',len(b['evidence'])<=FINAL_TOP_K))
    checks.append(('sufficiency field',isinstance(b.get('evidence_sufficiency'),dict)))
    checks.append(('sufficiency consistency',b.get('evidence_sufficiency')==assess_evidence_sufficiency(b['evidence'])))
    checks.append(('status consistency',b['retrieval_status']==('INSUFFICIENT_EVIDENCE' if b['evidence_sufficiency']['status']=='INSUFFICIENT_EVIDENCE' else 'READY_FOR_AGENTIC_GRAPHRAG')))
    for e in b['evidence']:
        checks.append(('temporal',utc(e['available_at'])<=asof))
        checks.append(('project',e['instance_id']==b['instance_id']))
        checks.append(('unique',e['evidence_id'] not in ids));ids.add(e['evidence_id'])
        checks.append(('source traceability',bool(e['node_uid'] or e['source_id'] or e['edge_uid'])))
        checks.append(('no future labels',not (set(e)&FORBIDDEN)))
        for step in e['graph_path']:
            edge_row=edges[edges.edge_uid==step['edge_uid']]
            checks.append(('graph edge exists',len(edge_row)==1))
            if len(edge_row)==1:
                checks.append(('graph edge available',edge_row.iloc[0]['available_at']<=asof))
                checks.append(('graph edge same project',edge_row.iloc[0]['instance_id']==b['instance_id']))
    return checks
report=[]
for b in pilot_bundles:
    c=validate_bundle(b)
    report.append({'trigger_id':b['trigger_id'],'checks':len(c),'failures':sum(not ok for _,ok in c),
                   'status':'PASS' if all(ok for _,ok in c) else 'FAIL'})
validation_df=pd.DataFrame(report);display(validation_df)
assert (validation_df.status=='PASS').all(), 'Retrieval validation failed; inspect evidence and paths'
print('ALL RETRIEVAL SAFETY CHECKS PASSED')

,trigger_id,checks,failures,status
0,aec0dacc1eac39d2943e,52,0,PASS
1,157e5547098d1ea35ccd,56,0,PASS
2,67b3c4b34422e9148001,66,0,PASS
3,0dbd32c9c1b0f80de3f6,66,0,PASS
4,ad8c84c5202d7c3bd855,66,0,PASS
...,...,...,...,...
134,a9c235efbd4e6012b92f,31,0,PASS
135,0209c192fc41fc695c00,46,0,PASS
136,019bf287333d68c8aa7c,41,0,PASS
137,1f3ceff8e5dc5146fbbc,46,0,PASS


ALL RETRIEVAL SAFETY CHECKS PASSED


## 11. Final full-queue export — separate directory

Exports one record per request, including insufficient-evidence cases, after validating every bundle. The queue and manifest are written atomically. Previous experiment directories are never touched. A successful export is **not** a claim that evidence supports the risk.

In [ ]:
# Full-queue consistency: no skipped, repeated, or mismatched requests.
assert len(pilot_bundles)==len(selected_requests)
assert len({b['trigger_id'] for b in pilot_bundles})==len(pilot_bundles)
expected={r['trigger_id']:(part,r) for part,r in selected_requests}
for b in pilot_bundles:
    assert b['trigger_id'] in expected
    part,r=expected[b['trigger_id']]
    assert b['partition']==part
    assert b['snapshot_id']==r['snapshot_id']
    assert b['instance_id']==r['instance_id']
    assert int(b['week_number'])==int(r['week_number'])
    assert utc(b['checkpoint_time'])==utc(r['checkpoint_time'])
    assert b['evidence_sufficiency']==assess_evidence_sufficiency(b['evidence'])
    assert all(ok for _,ok in validate_bundle(b))

queue_path=OUT/'agentic_graphrag_retrieval_queue.jsonl'
temp_queue=queue_path.with_name(queue_path.name+'.partial')
with open(temp_queue,'w',encoding='utf-8') as f:
    for b in pilot_bundles:
        f.write(json.dumps(b,ensure_ascii=False,allow_nan=False)+'\n')
os.replace(temp_queue,queue_path)
validation_df.to_csv(OUT/'retrieval_validation.csv',index=False)
quality_df.to_csv(OUT/'evidence_sufficiency_summary.csv',index=False)
contract={
 'schema_version':'3.2',
 'module':'Event-focused graph traversal + balanced FAISS + BM25 + RRF + cross-encoder',
 'input_queues':[str(p) for p in queue_files],
 'output_queue':str(queue_path),
 'output_bundle_dir':str(bundle_dir),
 'next_notebook':'Graph Planner -> Evidence Verification -> Event Timeline -> Grounded Explanation',
 'required_evidence_fields':['evidence_id','source_id','source_type','available_at','text','evidence_quality','graph_path','rerank_score'],
 'temporal_rule':'Only evidence available at or before checkpoint_time',
 'quality_rule':'Only VERIFIED_SOURCE_TEXT may be quoted as source text. Derived transitions are interval observations.',
 'sufficiency_rule':'INSUFFICIENT_EVIDENCE when no event/dependency/verified source text is retrieved; still export for follow-up.',
 'shap_rule':'SHAP can be ON_DEMAND; final agent must resolve or flag unavailable drivers.',
 'request_count':len(selected_requests),
 'full_queue':RUN_LIMIT is None,
 'caveat':'No relevance labels; rankings and causal support are not independently verified.'
}
save_json_atomic(OUT/'retrieval_contract.json',contract)
manifest_out={
 'schema_version':'3.2','processed_requests':len(pilot_bundles),
 'input_requests':len(requests),'all_selected_processed':len(pilot_bundles)==len(selected_requests),
 'full_queue_complete':RUN_LIMIT is None and len(pilot_bundles)==len(requests),
 'insufficient_evidence_count':sum(b['retrieval_status']=='INSUFFICIENT_EVIDENCE' for b in pilot_bundles),
 'validation_failures':int((validation_df['status']!='PASS').sum()),
 'output_queue':str(queue_path)
}
save_json_atomic(OUT/'retrieval_run_manifest.json',manifest_out)
with open(queue_path,encoding='utf-8') as f:
    written=[json.loads(line) for line in f if line.strip()]
assert len(written)==len(selected_requests)
assert [b['trigger_id'] for b in written]==[r['trigger_id'] for _,r in selected_requests]
assert manifest_out['validation_failures']==0
print('COMPLETE RETRIEVAL EXPORT VALIDATED:',len(written),'requests')
print('Full queue complete:',manifest_out['full_queue_complete'])
print('Insufficient-evidence requests:',manifest_out['insufficient_evidence_count'])
print('Output directory:',OUT)


COMPLETE RETRIEVAL EXPORT VALIDATED: 139 requests
Full queue complete: True
Insufficient-evidence requests: 14
Output directory: /content/drive/MyDrive/GC1/Temporal_snapshots/graph_hybrid_retrieval_event_focused_full_v2_output
